In [ ]:
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

!apt-get update -qq
!apt-get install fonts-nanum* -qq

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings(action='ignore')

path = '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf' # 나눔 고딕
font_name = fm.FontProperties(fname=path, size=10).get_name() # 기본 폰트 사이즈 : 10
plt.rc('font', family=font_name)

fm.fontManager.addfont(path)

#텍스트 전처리

### 1. 토큰화
### 2. 텍스트 정제(Cleaning)
### 3. 형태소 분석 및 품사 태깅
### 4. 바이트페어 변환


In [ ]:
# 📌 하는 일: Google Colab 환경에서 파일 업로드
# 🎯 목적: 로컬에 있는 데이터 파일을 Colab 환경으로 가져오기
from google.colab import files  # Google Colab의 파일 시스템 접근 모듈 임포트
uploaded = files.upload()       # 파일 업로드 대화상자 실행 및 파일 서버에 업로드

Saving nreview_mask.csv to nreview_mask (1).csv


In [ ]:
# 📌 하는 일: 데이터 불러오기 및 구조 확인
# 🎯 목적: CSV 파일을 데이터프레임으로 변환하고 내용 확인
import pandas as pd             # 데이터 분석 라이브러리 pandas 임포트
df = pd.read_csv("./nreview_mask.csv")  # CSV 파일 읽어서 데이터프레임 생성
df                              # 데이터프레임 내용 출력 (2180행 × 4열 구조 확인)

,id,date,text,rat
0,euge****,22.09.16.,새부리 KF94 마스크를 검색하니 네이버 랭킹 상위에 뜨고 가격도 착해서 호기심에 ...,평점5
1,gemm****,22.05.03.,컬러 KF94 마스크를 너무나도 기다렸어요\n예쁜 컬러에 벌크포장과 저렴하면서 퀄리...,평점5
2,gemm****,22.05.06.,컬러 KF94 마스크를 너무나도 기다렸어요\n세상 어디에도 없는 라이트실버 H워월V...,평점5
3,jina****,22.07.22.,라이브 보고 방송하는언니 쓴거 넘 예뻐서 100개 주문했는데 가로사이즈가 저한테 좀...,평점5
4,pim4****,22.10.21.,고민고민하다 디럭스 실버를 주문했는데\n진짜 이쁩니다. 과하지 않지만 평범하지도 않...,평점5
...,...,...,...,...
2175,goon****,22.05.10.,4중구조인 거에 비해 얇아요 그리고 귀끈이 진짜 편합니다 A사는 귀는 편하나 딱맞는...,평점5
2176,bboy****,23.04.04.,가격 싸서 좋아요 하지만\n흰색 마스크를 제외한 컬러가 들어간 마스크는\n코 부분 ...,평점4
2177,0019****,22.08.13.,우순 엄청큰 봉투가 와서 놀랏고 ㅋㅋㅋ\n마구마구 넣으신듯한... 상자에 구겨져들어...,평점3
2178,leej****,22.08.21.,기존에 에코브리즈 다른 마스크보다 우선 시원하고 크기도 큼직해서 개인적으로 기미라인...,평점5


## 1. 토큰화

사람이 텍스트를 이해 할 때는 문장 안의 단어를 보며 문장을 파악하고 이해를 합니다. 컴퓨터 또한 텍스트를 분석하기 위해서는 텍스트를 특정 의미 단위로 분할하여 문장을 파악할수 있게 합니다. 이러한 의미 단위을 토큰이라고 합니다.

### 문자로 토큰화

토큰화 방식중 가장 간단하게 접근하는 방식은 띄어쓰기를 기준으로 단어를 구분 하는 것입니다. 파이썬에서는 split() 함수를 이용하면 쉽게 토큰화가 가능합니다.

In [ ]:
# 📌 하는 일: 개별 텍스트 샘플 확인
# 🎯 목적: 실제 데이터 형태와 내용 파악하기
text = df.loc[0, 'text']        # 0번째 행의 'text' 컬럼 값 추출
text                            # 추출된 텍스트 출력 (첫 번째 리뷰 내용 확인)

'새부리 KF94 마스크를 검색하니 네이버 랭킹 상위에 뜨고 가격도 착해서 호기심에 주문해봤어요\n특히 핑크베이지 색상이 예뻐보여서요\n무난하게 맨날 화이트 블랙만 사보고 핑크 계열은 안사봐서 궁금하기도 하고 무엇보다 가격이 메리트가 커서 ㅋㅋ\n장바구니에 담을 때 분명 9,900원이었는데 결제하려고 보니까 12,900원 뜨더라구요? 그래서 주문 못하고 있다가 하루에도 라이브를 몇 번 씩 하시고 그 타이밍에 주문하면 9,900원이라 운좋게 아침 8시반 라방하기 전 8시에 주문 성공해서 당일 출고해서 담날 바로 받았어요\n봄 여름 핑크가 아니라 딱 가을 겨울 톤다운 핑크예요~\n살구핑크베이지랄까? 차분한 톤이라 오히려 맘에 드네요\n다른 후기처럼 중형인데 일반 대형 사이즈라길래 중형으로 주문했더니 역시나 넉넉하게 잘 맞아요\n남편이 써도 잘 맞더라구요\n웬만한 보통 얼굴 크기 남성이라면 중형도 OK일듯!\n가로폭이 커서 얼굴 전체를 잘 감싸주고 세로는 약간 짧은듯한데 그렇다고 부족하진 않아요\n사진에 미밋 대형, 아이랑 어른용과 비교 사이즈 남겼으니 참고 부탁드려요\n일단 25매 벌크 포장 뜯었을 때는 딱히 불량이 안보이고 괜찮은데 제발 한달 사용 후기로 평범 1점 주는 일이 없기를 바래야겠어요\n암튼 가성비 마스크로 추천!'

In [ ]:
# 📌 하는 일: 공백 기반 토크나이징 시도
# 🎯 목적: 가장 기본적인 단어 분리 방법 테스트
text = text.split(' ')          # 문자열을 공백 기준으로 분리하여 리스트 생성
print(text)                     # 분리된 단어 리스트 출력 (줄바꿈 문자 등 포함 확인)

['새부리', 'KF94', '마스크를', '검색하니', '네이버', '랭킹', '상위에', '뜨고', '가격도', '착해서', '호기심에', '주문해봤어요\n특히', '핑크베이지', '색상이', '예뻐보여서요\n무난하게', '맨날', '화이트', '블랙만', '사보고', '핑크', '계열은', '안사봐서', '궁금하기도', '하고', '무엇보다', '가격이', '메리트가', '커서', 'ㅋㅋ\n장바구니에', '담을', '때', '분명', '9,900원이었는데', '결제하려고', '보니까', '12,900원', '뜨더라구요?', '그래서', '주문', '못하고', '있다가', '하루에도', '라이브를', '몇', '번', '씩', '하시고', '그', '타이밍에', '주문하면', '9,900원이라', '운좋게', '아침', '8시반', '라방하기', '전', '8시에', '주문', '성공해서', '당일', '출고해서', '담날', '바로', '받았어요\n봄', '여름', '핑크가', '아니라', '딱', '가을', '겨울', '톤다운', '핑크예요~\n살구핑크베이지랄까?', '차분한', '톤이라', '오히려', '맘에', '드네요\n다른', '후기처럼', '중형인데', '일반', '대형', '사이즈라길래', '중형으로', '주문했더니', '역시나', '넉넉하게', '잘', '맞아요\n남편이', '써도', '잘', '맞더라구요\n웬만한', '보통', '얼굴', '크기', '남성이라면', '중형도', 'OK일듯!\n가로폭이', '커서', '얼굴', '전체를', '잘', '감싸주고', '세로는', '약간', '짧은듯한데', '그렇다고', '부족하진', '않아요\n사진에', '미밋', '대형,', '아이랑', '어른용과', '비교', '사이즈', '남겼으니', '참고', '부탁드려요\n일단', '25매', '벌크', '포장', '뜯었을', '때는', '딱히', '불량이', '안보이고', '괜찮은데', '제발', '한달', '사용', '후기로', '평범', '1점', 

### 숫자 인코딩 --> 벡터화

토큰화 이후 컴퓨터가 이를 분석 하기 위해선 문자로 나타나는 토큰을 벡터형태의 데이터로 바꾸어 주어야 합니다.    
토큰을 여러 숫자의 조합인 벡터로 최종 변환하기 이전에 토큰에 고유한 숫자를 부여하여 인코딩 과정을 선행 해주어야 합니다.   
*토큰화 => 숫자 인코딩 => 백터화*

- **인코딩**: 문자 토큰을 숫자 토큰으로 변환
- **디코딩**: 숫자 토큰을 문자 토큰으로 변환

기본적인 인코딩 방식은 모든 문자 토큰을 찾아 고유 번호를 맵핑시킨 **딕셔너리**를 만드는 방식입니다.


In [ ]:
# 📌 하는 일: 전체 데이터셋의 고유 단어 집합 생성
# 🎯 목적: 데이터에 있는 모든 고유 단어 수집하기
uq = set()                      # 중복 제거를 위한 빈 집합 생성
for text in df['text']:         # 모든 텍스트 데이터 순회
  sp = text.split(' ')          # 각 텍스트를 단어 리스트로 분리
  for w in sp:                  # 각 단어 순회
    uq.add(w)                   # 단어를 집합에 추가 (중복 자동 제거)
print(uq)                       # 완성된 고유 단어 집합 출력

# uq = set()
# for t in df['text']:
#     uq.update(t.split(' '))
# print(len(uq))

{'누락되었다가', '라이트실버컬러가', '착용이', '보풀처럼', '없습니다.\n색상은', '같기도한데', '받았어요!!\n\n아', '커요.', '교환신청했고', '있구요\n원래', '예뻐도', 'ㅋㅋ\n마스크', '빨간매직이', '평면형만', '색깔,', '특장점은', '더편해요ㅎㅎ', '속하는데', '마스크만이', '일단', '여성이', '필요하니까요', '해요)', '불편했는데\n새부리형은', '부담스럽지', '어른도', '안쓰게되지싶어서', '가로,세로가', '큰', '맞음시면', '굿굿~~', '판매되길요~^^', '교체', '살라했는데', '됐었는데', '봄까지는', '만족해용\n================\n핑크만', '까지.', '1차', '냄새...', 'size', '운동하면서', '달라붙어서', '뻔', '좋고~\n마지막', '이뻐서', '있지만', '드네요~^^;어쨌든', '대만족이에요!!!!!!', '짱인', '뜯으라고', '구매했네요~^^\n정말', '병원에서', '워낙에', '숨쉬면', '고민이었어요,,ㅋㅋㅋ', '정착했어요', '첫구매', '비움이랑', '소비자', '더좋네요', '구입했어요.\n저한테는', '툭', 'ㅋㅋㅋ\n암튼', '했는데요', '얇으니까', '라이브방송할때', '끊어집니다!!!', '한예슬', '했습니다.', '산호색', '주문해봤네요😊', '사용중이였는데사진과', '하겠지만..', '안정적이어', '바뀌어서', '필시', '아니에요ㅎㅎ', '납작해서', '브리즈와', '점심에', '몇시가', '줄이고,', '사용중입니다\n제가', '없었어요.\n\n장점', '안나요.', '만들어', '한사이즈', '것..디럭스는', '기분만', '품절', '착용감,', '해줘서', '끊겼네요ㅎ\n약속시간', '해서\n구매했어요.\n품질', '라운드베이직', '빳빳한건\n커버가', '좀있는데', '작은얼굴분들은', '라이트가', '얼굴크기에는', '전날밤에', '체격이고요', '50개중에', '알지만', '구매중', '

In [ ]:
# 📌 하는 일: 단어-인덱스 매핑 딕셔너리 생성
# 🎯 목적: 각 단어에 고유 번호를 부여하여 숫자로 변환 가능하게 하기
enc_dict = {}                   # 빈 딕셔너리 생성
for i, t in enumerate(uq):      # 고유 단어 집합을 인덱스와 함께 순회
  enc_dict[t] = i               # 단어를 키로, 인덱스를 값으로 저장
print(enc_dict)                 # 완성된 매핑 딕셔너리 출력

# enc_dict = {w: i for i, w in enumerate(uq)}
# print(enc_dict)

{'누락되었다가': 0, '라이트실버컬러가': 1, '착용이': 2, '보풀처럼': 3, '없습니다.\n색상은': 4, '같기도한데': 5, '받았어요!!\n\n아': 6, '커요.': 7, '교환신청했고': 8, '있구요\n원래': 9, '예뻐도': 10, 'ㅋㅋ\n마스크': 11, '빨간매직이': 12, '평면형만': 13, '색깔,': 14, '특장점은': 15, '더편해요ㅎㅎ': 16, '속하는데': 17, '마스크만이': 18, '일단': 19, '여성이': 20, '필요하니까요': 21, '해요)': 22, '불편했는데\n새부리형은': 23, '부담스럽지': 24, '어른도': 25, '안쓰게되지싶어서': 26, '가로,세로가': 27, '큰': 28, '맞음시면': 29, '굿굿~~': 30, '판매되길요~^^': 31, '교체': 32, '살라했는데': 33, '됐었는데': 34, '봄까지는': 35, '만족해용\n================\n핑크만': 36, '까지.': 37, '1차': 38, '냄새...': 39, 'size': 40, '운동하면서': 41, '달라붙어서': 42, '뻔': 43, '좋고~\n마지막': 44, '이뻐서': 45, '있지만': 46, '드네요~^^;어쨌든': 47, '대만족이에요!!!!!!': 48, '짱인': 49, '뜯으라고': 50, '구매했네요~^^\n정말': 51, '병원에서': 52, '워낙에': 53, '숨쉬면': 54, '고민이었어요,,ㅋㅋㅋ': 55, '정착했어요': 56, '첫구매': 57, '비움이랑': 58, '소비자': 59, '더좋네요': 60, '구입했어요.\n저한테는': 61, '툭': 62, 'ㅋㅋㅋ\n암튼': 63, '했는데요': 64, '얇으니까': 65, '라이브방송할때': 66, '끊어집니다!!!': 67, '한예슬': 68, '했습니다.': 69, '산호색': 70, '주문해봤네요😊': 71, '사용중이였는데사진과': 72, '하겠지만..': 73, '안정적이어': 7

In [ ]:
# 📌 하는 일: 실제 텍스트 인코딩 테스트
# 🎯 목적: 만든 딕셔너리로 텍스트를 숫자 시퀀스로 변환해보기
text = df.loc[0, 'text']        # 첫 번째 리뷰 텍스트 다시 추출
str_tk = text.split(' ')        # 텍스트를 단어 리스트로 분리
enc_tk = [enc_dict[w] for w in str_tk]  # 각 단어를 해당 인덱스로 변환
print(enc_tk)                   # 인코딩된 숫자 시퀀스 출력

[24701, 743, 119, 11755, 27427, 26979, 26927, 23904, 19897, 4375, 15315, 30101, 29344, 21056, 31902, 573, 30046, 4862, 21705, 28319, 10206, 7688, 27669, 16586, 16228, 6182, 31405, 20756, 2723, 2768, 18745, 5878, 9973, 9164, 4063, 31060, 16185, 17104, 20187, 17765, 11390, 21521, 16143, 12685, 17503, 16408, 7487, 11372, 7951, 12436, 8483, 30860, 20750, 31376, 29646, 11031, 9191, 20187, 27741, 1394, 3669, 12398, 15887, 8827, 7238, 22070, 18367, 29275, 3579, 20446, 17492, 16921, 28967, 394, 27780, 1589, 19302, 20846, 18619, 10865, 27956, 30880, 25429, 5959, 23169, 16300, 30199, 23974, 18706, 30199, 3645, 18099, 1626, 6985, 10317, 9912, 17729, 20756, 1626, 26840, 30199, 1407, 2871, 12425, 7965, 8426, 24668, 8425, 9652, 3138, 24282, 7973, 29172, 30687, 3934, 10799, 3228, 17840, 8110, 1854, 13774, 10501, 6967, 29601, 21910, 28952, 1472, 26129, 16657, 20735, 26058, 23049, 17240, 30126, 4727, 9819, 22366, 25467, 29494]


## 2. 텍스트 정제(클리닝)

기본적인 토큰화 결과 특수문자나 의미없는 단어들 또한 하나의 토큰으로 분리 되었습니다. 이러한 불필요한 토큰은 텍스트 분석에 있어 방해 요소가 됩니다.  

따라서 의미 없는 기호, 단어를 제거하는 작업을 먼저 진행하는 클렌징 작업이 필요합니다.

In [ ]:
# 📌 하는 일: 다른 샘플 텍스트 확인
# 🎯 목적: 다양한 데이터 형태 파악하기
text = df.loc[4, 'text']        # 5번째 행의 텍스트 추출
text                            # 추출된 텍스트 출력

'고민고민하다 디럭스 실버를 주문했는데\n진짜 이쁩니다. 과하지 않지만 평범하지도 않아 좋아요. 어두운 옷, 밝은 옷에도 쓰기 좋네요. 큰 얼굴이지만 디럭스 사이즈는 여유있게 편하고 씁니다. 꽉 끼는 불편함이 없어서 맘에 들어요.\n지인한테 써보라고 자신있게 1봉 나눔했어요~^^\n구매평 중 많았던 연결부위 끈 떨어짐 현상은 아직 없었어서 좋아라 했는데,\n몇일전 아침에 쓸려고 보니 끈 한쪽의 반이 잘려 나간 듯이 짧았어요. 살펴보니 다행이 하나만 그러네요.\n불량이 하필 실버 색상에서 나와서 그게 아쉬워요. 다른 색상 마스크 였다면 좋았을 텐데ㅠㅠ\n유색마스크는 좀 비싸던데, 가격대비 제품이 좋아서 만족도가 좋았어요.\n그래도 다음에는 불량체크 잘 된 제품으로 꼭받고 싶어요.\n흰색, 검정색 지겹고, 과한 색상은 부담이신분들께 조심스럽게 실버마스크 추천합니다.'

### 특수 문자 제거하기

특수 문자 제거는 파이썬 내장 모듈인 `re`모듈을 사용합니다
> re.sub(수정될 문자열의 정규식, 수정후의 문자열, 문자열)

\w: A–Z, a–z, 0–9, _

In [ ]:
### 특수 문자 제거하기
# 📌 하는 일: 정규표현식을 이용한 텍스트 정제
# 🎯 목적: 특수문자, 줄바꿈, 공백 등을 정리하여 깔끔한 텍스트 만들기
import re                       # 정규표현식 모듈 임포트
rtext = re.sub(r'[^\w\s]', '', text)  # 단어문자/공백 제외 모든 문자 제거
rtext = re.sub(r'[\n\t]', ' ', rtext)  # 줄바꿈/탭을 공백으로 치환
rtext = re.sub(r'\s+', ' ', rtext)     # 연속 공백을 하나로 축소
rtext                            # 정제된 텍스트 출력

# clean_text = re.sub(r'[^\w\s]', '', text)
# clean_text = re.sub(r'\s+', ' ', clean_text)
# print(clean_text)

'고민고민하다 디럭스 실버를 주문했는데 진짜 이쁩니다 과하지 않지만 평범하지도 않아 좋아요 어두운 옷 밝은 옷에도 쓰기 좋네요 큰 얼굴이지만 디럭스 사이즈는 여유있게 편하고 씁니다 꽉 끼는 불편함이 없어서 맘에 들어요 지인한테 써보라고 자신있게 1봉 나눔했어요 구매평 중 많았던 연결부위 끈 떨어짐 현상은 아직 없었어서 좋아라 했는데 몇일전 아침에 쓸려고 보니 끈 한쪽의 반이 잘려 나간 듯이 짧았어요 살펴보니 다행이 하나만 그러네요 불량이 하필 실버 색상에서 나와서 그게 아쉬워요 다른 색상 마스크 였다면 좋았을 텐데ㅠㅠ 유색마스크는 좀 비싸던데 가격대비 제품이 좋아서 만족도가 좋았어요 그래도 다음에는 불량체크 잘 된 제품으로 꼭받고 싶어요 흰색 검정색 지겹고 과한 색상은 부담이신분들께 조심스럽게 실버마스크 추천합니다'

### 불용어 제거

불용어 제거는 파이썬의 문자열 함수와 리스트, 반복문등을 이용하여 구성 가능합니다.    
먼저 토큰화 이후 반복문을 통해서 해당 단어가 불용어에 속하는지 `if` `in` 문을 활용하여 체크하고 속하지 않으면 새로운 리스트에 담아 `join`해줍니다.

In [ ]:
# 📌 하는 일: 정제된 텍스트 토크나이징 확인
# 🎯 목적: 정제 후 토크나이징 결과 비교하기
print(rtext.split(' '))         # 정제된 텍스트를 공백 기준으로 분리하여 출력

['고민고민하다', '디럭스', '실버를', '주문했는데', '진짜', '이쁩니다', '과하지', '않지만', '평범하지도', '않아', '좋아요', '어두운', '옷', '밝은', '옷에도', '쓰기', '좋네요', '큰', '얼굴이지만', '디럭스', '사이즈는', '여유있게', '편하고', '씁니다', '꽉', '끼는', '불편함이', '없어서', '맘에', '들어요', '지인한테', '써보라고', '자신있게', '1봉', '나눔했어요', '구매평', '중', '많았던', '연결부위', '끈', '떨어짐', '현상은', '아직', '없었어서', '좋아라', '했는데', '몇일전', '아침에', '쓸려고', '보니', '끈', '한쪽의', '반이', '잘려', '나간', '듯이', '짧았어요', '살펴보니', '다행이', '하나만', '그러네요', '불량이', '하필', '실버', '색상에서', '나와서', '그게', '아쉬워요', '다른', '색상', '마스크', '였다면', '좋았을', '텐데ㅠㅠ', '유색마스크는', '좀', '비싸던데', '가격대비', '제품이', '좋아서', '만족도가', '좋았어요', '그래도', '다음에는', '불량체크', '잘', '된', '제품으로', '꼭받고', '싶어요', '흰색', '검정색', '지겹고', '과한', '색상은', '부담이신분들께', '조심스럽게', '실버마스크', '추천합니다']


In [ ]:
# 토큰화 후 반복문을 활용
# 📌 하는 일: 불용어(Stopword) 제거
# 🎯 목적: 의미 분석에 도움이 안 되는 단어들 제거하기
sptext = rtext.split(' ')       # 정제된 텍스트를 단어 리스트로 분리
st_words = ['', '진짜', '않지만']  # 제거할 불용어 리스트 정의
nw = []                         # 새로운 단어 리스트용 빈 리스트 생성
for w in sptext:                # 원본 단어 리스트 순회
  if w not in st_words:         # 현재 단어가 불용어가 아닌지 확인
    nw.append(w)                # 불용어가 아니면 새 리스트에 추가
rtext = ' '.join(nw)            # 새 단어 리스트를 공백으로 연결하여 문자열 생성
rtext                           # 불용어 제거된 최종 텍스트 출력

# stopwords = ['', '진짜', '않지만', '마스크를']
# clean_tokens = [w for w in clean_text.split(' ') if w not in stopwords and len(w) > 1]
# print(clean_tokens)
# len(clean_tokens)

'고민고민하다 디럭스 실버를 주문했는데 이쁩니다 과하지 평범하지도 않아 좋아요 어두운 옷 밝은 옷에도 쓰기 좋네요 큰 얼굴이지만 디럭스 사이즈는 여유있게 편하고 씁니다 꽉 끼는 불편함이 없어서 맘에 들어요 지인한테 써보라고 자신있게 1봉 나눔했어요 구매평 중 많았던 연결부위 끈 떨어짐 현상은 아직 없었어서 좋아라 했는데 몇일전 아침에 쓸려고 보니 끈 한쪽의 반이 잘려 나간 듯이 짧았어요 살펴보니 다행이 하나만 그러네요 불량이 하필 실버 색상에서 나와서 그게 아쉬워요 다른 색상 마스크 였다면 좋았을 텐데ㅠㅠ 유색마스크는 좀 비싸던데 가격대비 제품이 좋아서 만족도가 좋았어요 그래도 다음에는 불량체크 잘 된 제품으로 꼭받고 싶어요 흰색 검정색 지겹고 과한 색상은 부담이신분들께 조심스럽게 실버마스크 추천합니다'

### 전체 데이터에 클렌징 적용

DataFrame 사용시 apply() 사용하여 일괄처리해줍니다


In [ ]:
# 📌 하는 일: 재사용 가능한 클렌징 함수 개발
# 🎯 목적: 전체 데이터셋에 일관된 전처리 적용하기
import re                       # 정규표현식 모듈 임포트
stopwords = ['', '진짜','않지만']  # 불용어 리스트 정의

def cleaning(review):
  review = re.sub(r'[^\w\s]', '', review)  # 특수문자 제거
  review = re.sub(r'[\n\t]', ' ', review)  # 줄바꿈/탭 제거
  review = re.sub(r'\s+', ' ', review)     # 연속 공백 제거
  review= review.strip()                   # 양끝 공백 제거
  # 불용어 제거
  review = ' '.join([w for w in review.split(' ') if w not in stopwords])
  # 한글자 제거
  review = ' '.join([w for w in review.split(' ') if len(w) > 1])
  return review                            # 정제된 텍스트 반환

# def cleaning(review):
#     review = re.sub(r'[^\w\s]', '', review)
#     review = re.sub(r'\s+', ' ', review)
#     review = review.strip()
#     return ' '.join([w for w in review.split(' ') if w not in stopwords and len(w) > 1])

# 클렌징 함수 적용
# 📌 하는 일: 전체 데이터셋에 전처리 적용
# 🎯 목적: 모든 리뷰 텍스트를 일괄 정제하기
df['clean'] = df['text'].apply(cleaning)   # 'text' 컬럼 전체에 cleaning 함수 적용
df[['text', 'clean']]                      # 원본과 정제된 텍스트 비교 출력

,text,clean
0,새부리 KF94 마스크를 검색하니 네이버 랭킹 상위에 뜨고 가격도 착해서 호기심에 ...,새부리 KF94 마스크를 검색하니 네이버 랭킹 상위에 뜨고 가격도 착해서 호기심에 ...
1,컬러 KF94 마스크를 너무나도 기다렸어요\n예쁜 컬러에 벌크포장과 저렴하면서 퀄리...,컬러 KF94 마스크를 너무나도 기다렸어요 예쁜 컬러에 벌크포장과 저렴하면서 퀄리티...
2,컬러 KF94 마스크를 너무나도 기다렸어요\n세상 어디에도 없는 라이트실버 H워월V...,컬러 KF94 마스크를 너무나도 기다렸어요 세상 어디에도 없는 라이트실버 H워월V ...
3,라이브 보고 방송하는언니 쓴거 넘 예뻐서 100개 주문했는데 가로사이즈가 저한테 좀...,라이브 보고 방송하는언니 쓴거 예뻐서 100개 주문했는데 가로사이즈가 저한테 커요ㅠ...
4,고민고민하다 디럭스 실버를 주문했는데\n진짜 이쁩니다. 과하지 않지만 평범하지도 않...,고민고민하다 디럭스 실버를 주문했는데 이쁩니다 과하지 평범하지도 않아 좋아요 어두운...
...,...,...
2175,4중구조인 거에 비해 얇아요 그리고 귀끈이 진짜 편합니다 A사는 귀는 편하나 딱맞는...,4중구조인 거에 비해 얇아요 그리고 귀끈이 편합니다 A사는 귀는 편하나 딱맞는 느낌...
2176,가격 싸서 좋아요 하지만\n흰색 마스크를 제외한 컬러가 들어간 마스크는\n코 부분 ...,가격 싸서 좋아요 하지만 흰색 마스크를 제외한 컬러가 들어간 마스크는 부분 안쪽이 ...
2177,우순 엄청큰 봉투가 와서 놀랏고 ㅋㅋㅋ\n마구마구 넣으신듯한... 상자에 구겨져들어...,우순 엄청큰 봉투가 와서 놀랏고 ㅋㅋㅋ 마구마구 넣으신듯한 상자에 구겨져들어가잇어 ...
2178,기존에 에코브리즈 다른 마스크보다 우선 시원하고 크기도 큼직해서 개인적으로 기미라인...,기존에 에코브리즈 다른 마스크보다 우선 시원하고 크기도 큼직해서 개인적으로 기미라인...


## (추가) 바이트페어 변환

**바이트페어 변환** 은 텍스트를 바이트단위(문자)로 묶어가며 빈도 확률에 따라 최적화된 토큰셋을 만들어 냅니다.

    안 + 녕 => 빈도가 많음 => 토큰O
    안 + 녕 + 하 => 빈도가 적음 => 토큰X
    안 + 녕 + 하 + 세 + 요 => 빈도 많음 => 토큰 O

따라서 형태소 분석과 다르게 가지고 있는 텍스트뭉치에 따라 토큰이 최적화 되어 나온다는 장점이 있습니다.    

### SentencePiece 모듈 활용

`sentencepiece` 모듈은 바이트페어 인코딩 기능이 있는 대표적인 파이썬 라이브러리입니다.   
sentencepiece 모듈을 활용하여 다음과정을 통해 토큰화가 가능합니다.
1. sentencepiece 모듈에서 SentencePieceTrainer 객체를 통해 텍스트뭉치에서 토큰셋(model) 파일을 만들어냅니다.   
2. sentencepiece 모듈의 SentencePieceProcessor객체에 생성한 토큰셋 파일을 변환기(인코더)를 만들어 냅니다.
3. 만든 인코더에서 encode() 함수를 활용하여 텍스트를 토큰화 합니다.

[sentencepiece](https://github.com/google/sentencepiece)

In [ ]:
# 📌 하는 일: SentencePiece 라이브러리 확인
# 🎯 목적: 토크나이저 라이브러리 설치 및 버전 확인
import sentencepiece as spm     # SentencePiece 라이브러리 임포트
print(spm.__version__)          # 설치된 버전 출력

0.2.1


### 텍스트 뭉치 데이터를 텍스트 파일로 생성

`sentencepiece`는 텍스트 파일로부터 토큰을 찾아내는 학습을 진행하기 때문에 먼저 가지고 있는 텍스트를 하나의 .txt파일에 쓰는 작업이 필요합니다.

In [ ]:
# 📌 하는 일: SentencePiece 학습용 데이터 파일 생성 (수정됨)
# 🎯 목적: 토크나이저 모델 학습을 위한 정제된 텍스트 파일 만들기
import sentencepiece as spm     # SentencePiece 라이브러리 임포트
import pandas as pd             # pandas 라이브러리 임포트
import re                       # 정규표현식 모듈 임포트

df = pd.read_csv("nreview_mask.csv")  # 데이터 다시 읽기

with open('nreview_mask.txt', 'w', encoding='utf-8') as f:  # 'w' 모드로 파일 열기 (기존 내용 덮어씀)
  for text in df['text']:         # 모든 텍스트 순회
        text = str(text)          # 문자열로 형식 변환
        text = re.sub(r'[^\w\s]', '', text)  # 특수문자 제거
        text = re.sub(r'[\n\t]', ' ', text)  # 줄바꿈/탭 제거
        text = re.sub(r'\s+', ' ', text)     # 연속 공백 제거
        text= text.strip()        # 양끝 공백 제거
        try:
            f.write(text+'\n')    # 정제된 텍스트 파일에 기록
        except:
                pass              # 오류 발생 시 무시하고 진행

### 텍스트 파일로부터 토큰 사전 및 sentencepiece 모델 작성

`SentencePieceTrainer` `train()`
* `input`: 텍스트 뭉치 파일 경로
* `model_prefix`: 출력 모델 파일 경로
* `vocab_size`: 찾아낼 총 토큰 개수(가지고 있는 텍스트 양에 따라 결정)


In [ ]:
# 📌 하는 일: 서브워드 토크나이저 모델 학습
# 🎯 목적: 더 스마트한 토크나이징을 위한 AI 모델 학습
import os                         # 운영체제 인터페이스 모듈 임포트 - 파일 시스템 작업을 위해 필요
os.makedirs('./model', exist_ok=True)  # 'model'이라는 이름의 디렉토리 생성, 이미 존재하면 무시 - 학습된 모델 저장을 위한 폴더 준비

spm.SentencePieceTrainer.train(    # SentencePiece 토크나이저 모델 학습 시작
    input='nreview_mask.txt',     # 학습에 사용할 텍스트 데이터 파일 지정 - 전처리된 한국어 리뷰 데이터
    model_prefix='./model/spm',   # 생성될 모델 파일의 이름 접두사 지정 - spm.model과 spm.vocab 파일이 생성됨
    vocab_size=2000               # 단어장 크기를 2000개로 설정 - 모델이 인식할 수 있는 최대 서브워드 개수
)

### sentencepiece 모델을 활용하여 토큰화

sentencepiece 모델이 만들어진 후, `SentencePieceProcessor` 객체를 통해 생성된 모델을 다시 프로그램에 로드합니다. 이후 `encode()` 함수를 사용하여 토큰화 합니다
* 0번 토큰은 학습되지 않은 토큰으로 설정



In [ ]:
# 📌 하는 일: 테스트용 텍스트 확인
# 🎯 목적: 토크나이징 테스트할 샘플 선택
df.loc[3, 'text']                # 4번째 행의 텍스트 출력

'라이브 보고 방송하는언니 쓴거 넘 예뻐서 100개 주문했는데 가로사이즈가 저한테 좀 커요ㅠㅠ\n두번째 사진은 볼부분을 밀착시키기위해 코와이어를 접어봤어용ㅋㅋ\n블로그나 리뷰,길거리를 다녀도 저렇게 쓰는사람 굉장히 많더라구요ㅠㅠ\n잘못된 마스크 쓰는법이에요\n코 접히는 부분은 최대한 콧등에 붙혀서 쓰세요ㅠㅠ\n저렇게 쓰면 마스크가 뭔 소용인가용?\n콧등에 붙히고 볼부분이 밀착이 안되면 마스크가 큰거라고 생각합니당\n아무튼 중형인데도 커서 넘 아쉽지만....ㅠ\n끈을 묶어쓰면 괜찮아서 귀찮지만 묶어서 써야겠네용\n색상이나 촉감은 넘 좋아요\n얼굴 닿는 안감도 부들부들~~\n공장냄새(?)도 안나고 마스크 자체 디자인도 넘 예뻐요!!!\n지금 이 중형보다 조금 작은 사이즈가 나왔으면 하는 바램...!!!'

In [ ]:
# 📌 하는 일: 학습된 토크나이저로 텍스트 변환 테스트
# 🎯 목적: 모델이 제대로 작동하는지 검증하기
import sentencepiece as spm       # SentencePiece 라이브러리 임포트
import pandas as pd               # pandas 라이브러리 임포트
import re                         # 정규표현식 모듈 임포트

sp = spm.SentencePieceProcessor(model_file='./model/spm.model')  # 학습된 모델 로드
df = pd.read_csv("nreview_mask.csv")        # 데이터 다시 읽기
text = df.loc[3, 'text']          # 테스트할 텍스트 추출
print(sp.encode(text, out_type=str))  # 텍스트를 서브워드 문자열로 인코딩하여 출력
print(sp.encode(text, out_type=int))  # 텍스트를 서브워드 인덱스로 인코딩하여 출력

['▁라이브', '▁보고', '▁방송', '하는', '언', '니', '▁', '쓴', '거', '▁넘', '▁예뻐서', '▁100', '개', '▁주문했는데', '▁가로', '사이즈가', '▁저한테', '▁좀', '▁커요', 'ᅲᅲ', '▁두', '번째', '▁사진', '은', '▁볼', '부분', '을', '▁밀착', '시', '키', '기', '위', '해', '▁코', '와', '이', '어', '를', '▁', '접', '어', '봤', '어', '용', 'ᄏᄏ', '▁', '블', '로', '그', '나', '▁리뷰', ',', '길', '거', '리', '를', '▁다', '녀', '도', '▁저', '렇', '게', '▁쓰는', '사', '람', '▁', '굉', '장', '히', '▁많', '더라구요', 'ᅲᅲ', '▁잘', '못', '된', '▁마스크', '▁쓰는', '법', '이에요', '▁코', '▁', '접', '히', '는', '▁부분', '은', '▁', '최', '대', '한', '▁', '콧', '등', '에', '▁붙', '혀', '서', '▁쓰', '세요', 'ᅲᅲ', '▁저', '렇', '게', '▁쓰면', '▁마스크가', '▁', '뭔', '▁소', '용', '인', '가', '용', '?', '▁', '콧', '등', '에', '▁붙', '히', '고', '▁볼', '부분이', '▁밀착', '이', '▁안되', '면', '▁마스크가', '▁큰', '거라', '고', '▁생각', '합', '니', '당', '▁아무', '튼', '▁중형인데', '도', '▁커서', '▁넘', '▁아', '쉽', '지만', '....', 'ᅲ', '▁끈을', '▁', '묶', '어', '쓰면', '▁괜찮', '아서', '▁귀', '찮', '지만', '▁묶어서', '▁써야', '겠', '네', '용', '▁색상이', '나', '▁', '촉', '감', '은', '▁넘', '▁좋아요', '▁얼굴', '▁닿는', '▁안감', '도', '▁부', '

> spm를 활용한 바이트페어 토큰화 결과중 `_`는 기존 문장의 띄어쓰기를 의미합니다
> 띄어쓰기로 join하여도 기본 문장의 띄어쓰기 정보를 읽어버리지 않습니다.


### 특수 토큰을 추가

`bos_id`:문장의 시작을 알리는 토큰 번호
`eos_id`:문장의 끝을 알리느 토큰 번호
`unk_id`:학습되지 않은 토큰 번호
`pad_id`: 패딩 토큰 번호(무시 토큰)

In [ ]:
# 📌 하는 일: 개선된 설정으로 모델 재학습
# 🎯 목적: 더 나은 성능을 위한 하이퍼파라미터 튜닝
spm.SentencePieceTrainer.train(input='nreview_mask.txt',  # 학습 데이터 파일
                            model_prefix='./model/spm',   # 출력 모델 파일명
                            vocab_size=4000,              # 단어장 크기 4000개로 증가
                            bos_id=1,                     # 문장 시작 토큰 ID 설정
                            eos_id=2,                     # 문장 끝 토큰 ID 설정
                            unk_id=1,                     # 알수없는 토큰 ID 설정
                            pad_id=0                      # 패딩 토큰 ID 설정
                            )

In [ ]:
# 📌 하는 일: 문장 단위 토크나이징 테스트
# 🎯 목적: 실제 응용 환경에서의 사용 방법 검증
sp = spm.SentencePieceProcessor(model_file='./model/spm.model')  # 새로운 모델 로드
df = pd.read_csv("nreview_mask.csv")        # 데이터 다시 읽기
text = df.loc[3, 'text']          # 테스트 텍스트 추출
for line in text.split('\n'):     # 텍스트를 줄바꿈 기준으로 분리하여 각 줄 처리
    print(sp.encode(line, out_type=str))  # 각 줄을 서브워드 문자열로 변환하여 출력
    print([sp.bos_id()] + sp.encode(line, out_type=int) + [sp.eos_id()])  # 각 줄을 인덱스로 변환하고 시작/끝 토큰 추가하여 출력

['▁라이브', '▁보고', '▁방송', '하는', '언', '니', '▁', '쓴', '거', '▁넘', '▁예뻐서', '▁100', '개', '▁주문했는데', '▁가로', '사이즈가', '▁저한테', '▁좀', '▁커요', 'ᅲᅲ']
[1, 211, 525, 301, 151, 1296, 83, 3, 989, 24, 127, 893, 425, 63, 574, 387, 922, 973, 25, 503, 340, 2]
['▁두', '번째', '▁사진', '은', '▁볼', '부분', '을', '▁밀착', '시', '키', '기', '위', '해', '▁코', '와', '이', '어', '를', '▁', '접', '어', '봤', '어', '용', 'ᄏᄏ']
[1, 250, 670, 128, 8, 279, 314, 15, 380, 78, 523, 29, 514, 62, 179, 105, 4, 32, 53, 3, 784, 32, 234, 32, 139, 352, 2]
['▁', '블', '로', '그', '나', '▁리뷰', ',', '길', '거', '리', '를', '▁다', '녀', '도', '▁저', '렇', '게', '▁쓰는', '사', '람', '▁', '굉', '장', '히', '▁많', '더라구요', 'ᅲᅲ']
[1, 3, 1290, 21, 305, 34, 420, 0, 172, 24, 56, 53, 41, 1280, 5, 108, 1989, 17, 443, 114, 1980, 3, 1920, 145, 42, 473, 146, 340, 2]
['▁잘', '못', '된', '▁마스크', '▁쓰는', '법', '이에요']
[1, 16, 1988, 207, 12, 443, 1505, 287, 2]
['▁코', '▁', '접', '히', '는', '▁부분', '은', '▁', '최', '대', '한', '▁', '콧', '등', '에', '▁붙', '혀', '서', '▁쓰', '세요', 'ᅲᅲ']
[1, 179, 3, 784, 42, 6, 456, 8, 3,